# STEP 21 — Structured Field Extraction Pipeline

This notebook converts document/OCR evidence into the canonical
BhumiGuard 19-field Saf Kabala schema.

Core rules:

- Blank/template documents are not treated as transaction records.
- OCR text is evidence, not automatically verified truth.
- Handwritten OCR fields require human verification.
- Every extracted field stores provenance and verification status.
- Only verified fields may enter the transaction-risk engine.

In [1]:
CORE_FIELDS = [
    "deed_id",
    "deed_type",
    "registration_office",
    "execution_date",

    "seller_id",
    "seller_name",
    "buyer_id",
    "buyer_name",

    "district",
    "upazila",
    "mouza",
    "survey_type",
    "khatian_no",
    "dag_no",

    "total_property_area",
    "transferred_area",
    "seller_ownership_share",

    "previous_deed_reference",
    "power_of_attorney_reference",
]

print(
    "Core fields:",
    len(CORE_FIELDS)
)

Core fields: 19


In [2]:
FIELD_GROUPS = {
    "deed": [
        "deed_id",
        "deed_type",
        "registration_office",
        "execution_date",
    ],

    "parties": [
        "seller_id",
        "seller_name",
        "buyer_id",
        "buyer_name",
    ],

    "parcel": [
        "district",
        "upazila",
        "mouza",
        "survey_type",
        "khatian_no",
        "dag_no",
    ],

    "ownership": [
        "total_property_area",
        "transferred_area",
        "seller_ownership_share",
    ],

    "history": [
        "previous_deed_reference",
        "power_of_attorney_reference",
    ],
}

In [3]:
sum(
    len(fields)
    for fields in FIELD_GROUPS.values()
)

19

In [4]:
ALLOWED_SOURCE_TYPES = [
    "PRINTED_TEXT_LAYER",
    "PRINTED_OCR",
    "HANDWRITTEN_OCR",
    "HUMAN_ENTRY",
    "DERIVED",
]

In [5]:
FIELD_VERIFICATION_STATUSES = [
    "NOT_FOUND",
    "OCR_CANDIDATE",
    "ANCHOR_ONLY",
    "HUMAN_VERIFIED",
    "DERIVED_FROM_VERIFIED_FIELDS",
]

In [7]:
def empty_field_record(
    field_name
):
    return {
        "field_name":
            field_name,

        "value":
            None,

        "source_type":
            None,

        "source_document":
            None,

        "source_page":
            None,

        "ocr_confidence":
            None,

        "verification_status":
            "NOT_FOUND",

        "review_required":
            False,

        "notes":
            None,
    }

In [8]:
empty_field_record(
    "khatian_no"
)

{'field_name': 'khatian_no',
 'value': None,
 'source_type': None,
 'source_document': None,
 'source_page': None,
 'ocr_confidence': None,
 'verification_status': 'NOT_FOUND',
 'review_required': False,
 'notes': None}

In [9]:
def create_empty_extraction_record(
    document_id
):

    return {
        "document_id":
            document_id,

        "document_status":
            "UNASSESSED",

        "fields": {
            field_name:
                empty_field_record(
                    field_name
                )

            for field_name
            in CORE_FIELDS
        },
    }

In [10]:
saf_kabala_record = (
    create_empty_extraction_record(
        "downloaded_saf_kabala_template"
    )
)

print(
    "Fields created:",
    len(
        saf_kabala_record[
            "fields"
        ]
    )
)

Fields created: 19


In [11]:
saf_kabala_record[
    "document_status"
] = (
    "TEMPLATE_LIKE_UNFILLED"
)

In [12]:
saf_kabala_record[
    "fields"
][
    "deed_type"
].update(
    {
        "value":
            "SAF_KABALA",

        "source_type":
            "PRINTED_OCR",

        "source_document":
            "সাফ-কবলা.pdf",

        "source_page":
            1,

        "verification_status":
            "HUMAN_VERIFIED",

        "review_required":
            False,

        "notes":
            "Document type verified from template heading.",
    }
)

In [13]:
def add_field_candidate(
    record,
    field_name,
    value,
    source_type,
    source_document,
    source_page=None,
    ocr_confidence=None,
    notes=None,
):

    if field_name not in CORE_FIELDS:
        raise ValueError(
            f"Unknown field: {field_name}"
        )

    field_record = (
        record[
            "fields"
        ][
            field_name
        ]
    )

    field_record[
        "value"
    ] = value

    field_record[
        "source_type"
    ] = source_type

    field_record[
        "source_document"
    ] = source_document

    field_record[
        "source_page"
    ] = source_page

    field_record[
        "ocr_confidence"
    ] = ocr_confidence

    field_record[
        "notes"
    ] = notes

    if (
        source_type
        == "HANDWRITTEN_OCR"
    ):

        field_record[
            "verification_status"
        ] = "OCR_CANDIDATE"

        field_record[
            "review_required"
        ] = True

    else:

        field_record[
            "verification_status"
        ] = "OCR_CANDIDATE"

        field_record[
            "review_required"
        ] = True

In [14]:
def verify_field(
    record,
    field_name,
    verified_value=None,
    notes=None,
):

    field_record = (
        record[
            "fields"
        ][
            field_name
        ]
    )

    if verified_value is not None:

        field_record[
            "value"
        ] = verified_value

    field_record[
        "verification_status"
    ] = "HUMAN_VERIFIED"

    field_record[
        "review_required"
    ] = False

    if notes is not None:

        field_record[
            "notes"
        ] = notes

In [15]:
def field_allowed_for_risk_engine(
    field_record
):

    return (
        field_record[
            "verification_status"
        ]
        in [
            "HUMAN_VERIFIED",
            "DERIVED_FROM_VERIFIED_FIELDS",
        ]
    )

In [16]:
print(
    "Deed type allowed:",
    field_allowed_for_risk_engine(
        saf_kabala_record[
            "fields"
        ][
            "deed_type"
        ]
    )
)

print(
    "Khatian allowed:",
    field_allowed_for_risk_engine(
        saf_kabala_record[
            "fields"
        ][
            "khatian_no"
        ]
    )
)

Deed type allowed: True
Khatian allowed: False


In [17]:
def document_allowed_for_transaction_engine(
    record
):

    blocked_statuses = {
        "TEMPLATE_LIKE_UNFILLED",
        "NON_DEED",
        "OCR_UNVERIFIED",
    }

    return (
        record[
            "document_status"
        ]
        not in blocked_statuses
    )

In [18]:
print(
    "Saf Kabala template allowed:",
    document_allowed_for_transaction_engine(
        saf_kabala_record
    )
)

Saf Kabala template allowed: False


In [19]:
def extraction_summary(
    record
):

    fields = list(
        record[
            "fields"
        ].values()
    )

    return {
        "document_id":
            record[
                "document_id"
            ],

        "document_status":
            record[
                "document_status"
            ],

        "total_fields":
            len(fields),

        "human_verified":
            sum(
                field[
                    "verification_status"
                ]
                == "HUMAN_VERIFIED"

                for field in fields
            ),

        "ocr_candidates":
            sum(
                field[
                    "verification_status"
                ]
                == "OCR_CANDIDATE"

                for field in fields
            ),

        "not_found":
            sum(
                field[
                    "verification_status"
                ]
                == "NOT_FOUND"

                for field in fields
            ),

        "review_required":
            sum(
                field[
                    "review_required"
                ]

                for field in fields
            ),

        "transaction_engine_allowed":
            document_allowed_for_transaction_engine(
                record
            ),
    }

In [20]:
saf_kabala_summary = (
    extraction_summary(
        saf_kabala_record
    )
)

saf_kabala_summary

{'document_id': 'downloaded_saf_kabala_template',
 'document_status': 'TEMPLATE_LIKE_UNFILLED',
 'total_fields': 19,
 'human_verified': 1,
 'ocr_candidates': 0,
 'not_found': 18,
 'review_required': 0,
 'transaction_engine_allowed': False}

In [21]:
print("STEP 21A SUMMARY")
print("----------------")

print(
    "Core fields:",
    len(
        CORE_FIELDS
    )
)

print(
    "Field groups:",
    len(
        FIELD_GROUPS
    )
)

print(
    "Saf Kabala status:",
    saf_kabala_record[
        "document_status"
    ]
)

print(
    "Human-verified fields:",
    saf_kabala_summary[
        "human_verified"
    ]
)

print(
    "Not-found fields:",
    saf_kabala_summary[
        "not_found"
    ]
)

print(
    "Template accepted by transaction engine:",
    saf_kabala_summary[
        "transaction_engine_allowed"
    ]
)

print(
    "\nCanonical Extraction Record: READY"
)

STEP 21A SUMMARY
----------------
Core fields: 19
Field groups: 5
Saf Kabala status: TEMPLATE_LIKE_UNFILLED
Human-verified fields: 1
Not-found fields: 18
Template accepted by transaction engine: False

Canonical Extraction Record: READY


# STEP 21B — Saf Kabala Field Extraction Specification

This step converts the verified Saf Kabala page analysis into a
field-level extraction specification.

Important rules:

- A field label is not treated as a field value.
- Narrative occurrences of words such as খতিয়ান or দাগ are not values.
- Party IDs must not be inferred from names.
- Missing values remain missing.
- OCR candidates require verification before use by the risk engine.

In [22]:
field_extraction_spec = [
    {
        "field_name": "deed_id",
        "pages": [1],
        "region": "HEADER",
        "evidence": "VERIFIED_LAYOUT",
        "method": "LABEL_VALUE",
        "value_rule": "EXTRACT_ONLY_IF_EXPLICIT_DEED_NUMBER_VALUE_EXISTS",
    },

    {
        "field_name": "deed_type",
        "pages": [1],
        "region": "TITLE_HEADER",
        "evidence": "VERIFIED_VALUE",
        "method": "DOCUMENT_CLASSIFICATION",
        "value_rule": "SAF_KABALA_HEADING",
    },

    {
        "field_name": "registration_office",
        "pages": [1],
        "region": "HEADER_SUMMARY",
        "evidence": "VERIFIED_LABEL",
        "method": "LABEL_VALUE",
        "value_rule": "DO_NOT_ACCEPT_LABEL_WITHOUT_VALUE",
    },

    {
        "field_name": "execution_date",
        "pages": [9],
        "region": "EXECUTION_SECTION",
        "evidence": "PROVISIONAL_LAYOUT",
        "method": "DATE_EXTRACTION",
        "value_rule": "HUMAN_VERIFY_DATE",
    },

    {
        "field_name": "seller_id",
        "pages": [3, 4, 5],
        "region": "SELLER_IDENTITY_BLOCK",
        "evidence": "VERIFIED_IDENTITY_SECTION",
        "method": "IDENTITY_FIELD",
        "value_rule": "EXPLICIT_ID_ONLY_DO_NOT_INFER_FROM_NAME",
    },

    {
        "field_name": "seller_name",
        "pages": [3, 4, 5],
        "region": "SELLER_IDENTITY_BLOCK",
        "evidence": "VERIFIED_IDENTITY_SECTION",
        "method": "IDENTITY_FIELD",
        "value_rule": "EXTRACT_FROM_SELLER_IDENTITY_BLOCK",
    },

    {
        "field_name": "buyer_id",
        "pages": [2],
        "region": "BUYER_IDENTITY_BLOCK",
        "evidence": "VERIFIED_IDENTITY_SECTION",
        "method": "IDENTITY_FIELD",
        "value_rule": "EXPLICIT_ID_ONLY_DO_NOT_INFER_FROM_NAME",
    },

    {
        "field_name": "buyer_name",
        "pages": [2],
        "region": "BUYER_IDENTITY_BLOCK",
        "evidence": "VERIFIED_IDENTITY_SECTION",
        "method": "IDENTITY_FIELD",
        "value_rule": "EXTRACT_FROM_BUYER_IDENTITY_BLOCK",
    },

    {
        "field_name": "district",
        "pages": [1],
        "region": "SUMMARY",
        "evidence": "VERIFIED_LABEL",
        "method": "LABEL_VALUE",
        "value_rule": "DO_NOT_ACCEPT_LABEL_WITHOUT_VALUE",
    },

    {
        "field_name": "upazila",
        "pages": [1],
        "region": "SUMMARY",
        "evidence": "VERIFIED_LABEL",
        "method": "LABEL_VALUE",
        "value_rule": "DO_NOT_ACCEPT_LABEL_WITHOUT_VALUE",
    },

    {
        "field_name": "mouza",
        "pages": [1, 11, 12],
        "region": "SUMMARY_OR_PROPERTY_SCHEDULE",
        "evidence": "VERIFIED_LAYOUT",
        "method": "LABEL_VALUE",
        "value_rule": "PREFER_PROPERTY_SCHEDULE_VALUE",
    },

    {
        "field_name": "survey_type",
        "pages": [11],
        "region": "PROPERTY_SCHEDULE",
        "evidence": "PROVISIONAL_LAYOUT",
        "method": "SCHEDULE_FIELD",
        "value_rule": "ACCEPT_ONLY_EXPLICIT_SURVEY_TEXT",
    },

    {
        "field_name": "khatian_no",
        "pages": [11],
        "region": "PROPERTY_SCHEDULE",
        "evidence": "VERIFIED_LABEL",
        "method": "SCHEDULE_FIELD",
        "value_rule": "LABEL_FOUND_IS_NOT_VALUE_FOUND",
    },

    {
        "field_name": "dag_no",
        "pages": [11],
        "region": "PROPERTY_SCHEDULE",
        "evidence": "VERIFIED_LABEL",
        "method": "SCHEDULE_FIELD",
        "value_rule": "LABEL_FOUND_IS_NOT_VALUE_FOUND",
    },

    {
        "field_name": "total_property_area",
        "pages": [11, 12],
        "region": "PROPERTY_SCHEDULE",
        "evidence": "PROVISIONAL_LAYOUT",
        "method": "AREA_EXTRACTION",
        "value_rule": "REQUIRE_NUMBER_AND_AREA_UNIT_CONTEXT",
    },

    {
        "field_name": "transferred_area",
        "pages": [1, 11, 12],
        "region": "SUMMARY_OR_PROPERTY_SCHEDULE",
        "evidence": "VERIFIED_LAYOUT",
        "method": "AREA_EXTRACTION",
        "value_rule": "REQUIRE_NUMBER_AND_AREA_UNIT_CONTEXT",
    },

    {
        "field_name": "seller_ownership_share",
        "pages": [9, 10],
        "region": "SHARE_OR_EXECUTION_SECTION",
        "evidence": "VERIFIED_LAYOUT",
        "method": "SHARE_EXTRACTION",
        "value_rule": "DO_NOT_DERIVE_WITHOUT_EXPLICIT_SHARE_EVIDENCE",
    },

    {
        "field_name": "previous_deed_reference",
        "pages": [7],
        "region": "OWNERSHIP_HISTORY",
        "evidence": "VERIFIED_LAYOUT",
        "method": "REFERENCE_EXTRACTION",
        "value_rule": "EXPLICIT_PREVIOUS_DEED_REFERENCE_ONLY",
    },

    {
        "field_name": "power_of_attorney_reference",
        "pages": [7, 9],
        "region": "POA_SECTION",
        "evidence": "VERIFIED_LAYOUT",
        "method": "REFERENCE_EXTRACTION",
        "value_rule": "DO_NOT_CONFUSE_POA_PARTIES_WITH_BUYER_SELLER",
    },
]

In [23]:
import pandas as pd

field_spec_df = pd.DataFrame(
    field_extraction_spec
)

field_spec_df

,field_name,pages,region,evidence,method,value_rule
0,deed_id,[1],HEADER,VERIFIED_LAYOUT,LABEL_VALUE,EXTRACT_ONLY_IF_EXPLICIT_DEED_NUMBER_VALUE_EXISTS
1,deed_type,[1],TITLE_HEADER,VERIFIED_VALUE,DOCUMENT_CLASSIFICATION,SAF_KABALA_HEADING
2,registration_office,[1],HEADER_SUMMARY,VERIFIED_LABEL,LABEL_VALUE,DO_NOT_ACCEPT_LABEL_WITHOUT_VALUE
3,execution_date,[9],EXECUTION_SECTION,PROVISIONAL_LAYOUT,DATE_EXTRACTION,HUMAN_VERIFY_DATE
4,seller_id,"[3, 4, 5]",SELLER_IDENTITY_BLOCK,VERIFIED_IDENTITY_SECTION,IDENTITY_FIELD,EXPLICIT_ID_ONLY_DO_NOT_INFER_FROM_NAME
5,seller_name,"[3, 4, 5]",SELLER_IDENTITY_BLOCK,VERIFIED_IDENTITY_SECTION,IDENTITY_FIELD,EXTRACT_FROM_SELLER_IDENTITY_BLOCK
6,buyer_id,[2],BUYER_IDENTITY_BLOCK,VERIFIED_IDENTITY_SECTION,IDENTITY_FIELD,EXPLICIT_ID_ONLY_DO_NOT_INFER_FROM_NAME
7,buyer_name,[2],BUYER_IDENTITY_BLOCK,VERIFIED_IDENTITY_SECTION,IDENTITY_FIELD,EXTRACT_FROM_BUYER_IDENTITY_BLOCK
8,district,[1],SUMMARY,VERIFIED_LABEL,LABEL_VALUE,DO_NOT_ACCEPT_LABEL_WITHOUT_VALUE
9,upazila,[1],SUMMARY,VERIFIED_LABEL,LABEL_VALUE,DO_NOT_ACCEPT_LABEL_WITHOUT_VALUE


In [24]:
mapped_fields = set(
    field_spec_df[
        "field_name"
    ]
)

core_field_set = set(
    CORE_FIELDS
)

missing_from_spec = (
    core_field_set
    - mapped_fields
)

extra_in_spec = (
    mapped_fields
    - core_field_set
)

print(
    "Mapped fields:",
    len(mapped_fields)
)

print(
    "Missing fields:",
    sorted(
        missing_from_spec
    )
)

print(
    "Unexpected fields:",
    sorted(
        extra_in_spec
    )
)

Mapped fields: 19
Missing fields: []
Unexpected fields: []


In [25]:
duplicate_field_specs = (
    field_spec_df[
        "field_name"
    ]
    .duplicated()
    .sum()
)

print(
    "Duplicate field specifications:",
    duplicate_field_specs
)

Duplicate field specifications: 0


In [26]:
invalid_page_refs = []

for row in field_spec_df.itertuples():

    for page in row.pages:

        if not (
            1 <= page <= 17
        ):

            invalid_page_refs.append(
                (
                    row.field_name,
                    page
                )
            )

print(
    "Invalid page references:",
    invalid_page_refs
)

Invalid page references: []


In [27]:
provisional_fields_df = (
    field_spec_df[
        field_spec_df[
            "evidence"
        ]
        == "PROVISIONAL_LAYOUT"
    ][
        [
            "field_name",
            "pages",
            "region",
            "value_rule"
        ]
    ]
)

provisional_fields_df

,field_name,pages,region,value_rule
3,execution_date,[9],EXECUTION_SECTION,HUMAN_VERIFY_DATE
11,survey_type,[11],PROPERTY_SCHEDULE,ACCEPT_ONLY_EXPLICIT_SURVEY_TEXT
14,total_property_area,"[11, 12]",PROPERTY_SCHEDULE,REQUIRE_NUMBER_AND_AREA_UNIT_CONTEXT


In [28]:
identity_fields_df = (
    field_spec_df[
        field_spec_df[
            "method"
        ]
        == "IDENTITY_FIELD"
    ][
        [
            "field_name",
            "pages",
            "region",
            "value_rule"
        ]
    ]
)

identity_fields_df

,field_name,pages,region,value_rule
4,seller_id,"[3, 4, 5]",SELLER_IDENTITY_BLOCK,EXPLICIT_ID_ONLY_DO_NOT_INFER_FROM_NAME
5,seller_name,"[3, 4, 5]",SELLER_IDENTITY_BLOCK,EXTRACT_FROM_SELLER_IDENTITY_BLOCK
6,buyer_id,[2],BUYER_IDENTITY_BLOCK,EXPLICIT_ID_ONLY_DO_NOT_INFER_FROM_NAME
7,buyer_name,[2],BUYER_IDENTITY_BLOCK,EXTRACT_FROM_BUYER_IDENTITY_BLOCK


In [29]:
property_schedule_fields = [
    "mouza",
    "survey_type",
    "khatian_no",
    "dag_no",
    "total_property_area",
    "transferred_area",
]

property_schedule_spec_df = (
    field_spec_df[
        field_spec_df[
            "field_name"
        ].isin(
            property_schedule_fields
        )
    ][
        [
            "field_name",
            "pages",
            "method",
            "evidence",
            "value_rule"
        ]
    ]
)

property_schedule_spec_df

,field_name,pages,method,evidence,value_rule
10,mouza,"[1, 11, 12]",LABEL_VALUE,VERIFIED_LAYOUT,PREFER_PROPERTY_SCHEDULE_VALUE
11,survey_type,[11],SCHEDULE_FIELD,PROVISIONAL_LAYOUT,ACCEPT_ONLY_EXPLICIT_SURVEY_TEXT
12,khatian_no,[11],SCHEDULE_FIELD,VERIFIED_LABEL,LABEL_FOUND_IS_NOT_VALUE_FOUND
13,dag_no,[11],SCHEDULE_FIELD,VERIFIED_LABEL,LABEL_FOUND_IS_NOT_VALUE_FOUND
14,total_property_area,"[11, 12]",AREA_EXTRACTION,PROVISIONAL_LAYOUT,REQUIRE_NUMBER_AND_AREA_UNIT_CONTEXT
15,transferred_area,"[1, 11, 12]",AREA_EXTRACTION,VERIFIED_LAYOUT,REQUIRE_NUMBER_AND_AREA_UNIT_CONTEXT


In [30]:
def extraction_candidate_allowed(
    value,
    source_type,
    verification_status
):

    if value is None:
        return False

    if verification_status in {
        "HUMAN_VERIFIED",
        "DERIVED_FROM_VERIFIED_FIELDS",
    }:
        return True

    return False

In [31]:
print(
    "Unverified OCR:",
    extraction_candidate_allowed(
        "123",
        "HANDWRITTEN_OCR",
        "OCR_CANDIDATE"
    )
)

print(
    "Human verified:",
    extraction_candidate_allowed(
        "123",
        "HANDWRITTEN_OCR",
        "HUMAN_VERIFIED"
    )
)

Unverified OCR: False
Human verified: True


In [32]:
def label_only_result(
    field_name,
    label_found,
    value_found
):

    if (
        label_found
        and not value_found
    ):
        return {
            "field_name":
                field_name,

            "value":
                None,

            "verification_status":
                "ANCHOR_ONLY",

            "review_required":
                False,
        }

    return None

In [33]:
label_only_result(
    field_name="khatian_no",
    label_found=True,
    value_found=False
)

{'field_name': 'khatian_no',
 'value': None,
 'verification_status': 'ANCHOR_ONLY',
 'review_required': False}

In [34]:
DERIVED_FIELDS = [
    "parcel_key",
    "portion_id",
]

DERIVED_FIELDS

['parcel_key', 'portion_id']

In [35]:
derived_field_rules = {
    "parcel_key":
        "district + upazila + mouza + survey_type + khatian_no + dag_no",

    "portion_id":
        "schedule + boundary + share evidence when available",
}

derived_field_rules

{'parcel_key': 'district + upazila + mouza + survey_type + khatian_no + dag_no',
 'portion_id': 'schedule + boundary + share evidence when available'}

In [36]:
print("STEP 21B SUMMARY")
print("----------------")

print(
    "Core fields:",
    len(
        CORE_FIELDS
    )
)

print(
    "Mapped extraction fields:",
    len(
        field_spec_df
    )
)

print(
    "Missing specifications:",
    len(
        missing_from_spec
    )
)

print(
    "Duplicate specifications:",
    duplicate_field_specs
)

print(
    "Invalid page references:",
    len(
        invalid_page_refs
    )
)

print(
    "Provisional-layout fields:",
    len(
        provisional_fields_df
    )
)

print(
    "Identity fields:",
    len(
        identity_fields_df
    )
)

print(
    "Derived fields:",
    len(
        DERIVED_FIELDS
    )
)

print(
    "\nField Extraction Specification: READY"
)

STEP 21B SUMMARY
----------------
Core fields: 19
Mapped extraction fields: 19
Missing specifications: 0
Duplicate specifications: 0
Invalid page references: 0
Provisional-layout fields: 3
Identity fields: 4
Derived fields: 2

Field Extraction Specification: READY


# STEP 21C — Conservative Field Extraction Engine v1

This step implements the first conservative field extractor.

Design principles:

- Field labels and field values are separate evidence.
- Narrative mentions are never treated as structured values.
- Numbers are accepted only when explicitly attached to the expected label.
- Area values require both a number and an area unit.
- Extracted OCR values remain candidates until human verification.
- Blank Saf Kabala templates must not generate false transaction values.

In [37]:
import re
from copy import deepcopy

In [38]:
BENGALI_DIGITS = str.maketrans(
    "০১২৩৪৫৬৭৮৯",
    "0123456789"
)


def normalize_digits(text):
    if text is None:
        return None

    return str(text).translate(
        BENGALI_DIGITS
    )

In [39]:
print(
    normalize_digits(
        "খতিয়ান নং: ১২৩"
    )
)

খতিয়ান নং: 123


In [40]:
def clean_ocr_text(text):

    if text is None:
        return ""

    text = str(text)

    text = text.replace(
        "\r",
        "\n"
    )

    text = re.sub(
        r"[ \t]+",
        " ",
        text
    )

    return text.strip()

In [41]:
def extraction_result(
    field_name,
    anchor_found=False,
    value=None,
    method=None,
    evidence_text=None,
):

    if value is not None:

        status = "OCR_CANDIDATE"

    elif anchor_found:

        status = "ANCHOR_ONLY"

    else:

        status = "NOT_FOUND"

    return {
        "field_name":
            field_name,

        "anchor_found":
            anchor_found,

        "value":
            value,

        "status":
            status,

        "method":
            method,

        "evidence_text":
            evidence_text,
    }

In [42]:
def get_lines(
    text
):

    cleaned = clean_ocr_text(
        text
    )

    return [
        line.strip()
        for line in cleaned.split("\n")
        if line.strip()
    ]

In [43]:
def extract_labeled_number(
    text,
    field_name,
    labels
):

    lines = get_lines(
        text
    )

    anchor_found = False

    for line in lines:

        normalized_line = (
            normalize_digits(
                line
            )
        )

        for label in labels:

            if label not in line:
                continue

            anchor_found = True

            escaped_label = (
                re.escape(
                    label
                )
            )

            pattern = (
                escaped_label
                +
                r"\s*"
                r"(?:নং|নম্বর|নং\.|no\.?)?"
                r"\s*"
                r"[:ঃ\-–]?\s*"
                r"([0-9]+)"
            )

            match = re.search(
                pattern,
                normalized_line,
                flags=re.IGNORECASE
            )

            if match:

                return extraction_result(
                    field_name=
                        field_name,

                    anchor_found=
                        True,

                    value=
                        match.group(1),

                    method=
                        "LABEL_NUMBER",

                    evidence_text=
                        line,
                )

    return extraction_result(
        field_name=
            field_name,

        anchor_found=
            anchor_found,

        method=
            "LABEL_NUMBER",
    )

In [44]:
narrative_test = """
প্রকাশ থাকে যে, অত্র দলিলের দাগ, খতিয়ান,
মৌজা ও অন্যান্য তথ্য সম্পর্কে ভুল থাকিলে...
"""

result = extract_labeled_number(
    narrative_test,
    field_name="khatian_no",
    labels=[
        "খতিয়ান",
        "খতিয়ান",
    ]
)

result

{'field_name': 'khatian_no',
 'anchor_found': True,
 'value': None,
 'status': 'ANCHOR_ONLY',
 'method': 'LABEL_NUMBER',
 'evidence_text': None}

In [45]:
positive_khatian_test = """
খতিয়ান নং: ১২৩
"""

extract_labeled_number(
    positive_khatian_test,
    field_name="khatian_no",
    labels=[
        "খতিয়ান",
        "খতিয়ান",
    ]
)

{'field_name': 'khatian_no',
 'anchor_found': True,
 'value': '123',
 'status': 'OCR_CANDIDATE',
 'method': 'LABEL_NUMBER',
 'evidence_text': 'খতিয়ান নং: ১২৩'}

In [46]:
positive_dag_test = """
দাগ নং: ৪৫৬
"""

extract_labeled_number(
    positive_dag_test,
    field_name="dag_no",
    labels=[
        "দাগ"
    ]
)

{'field_name': 'dag_no',
 'anchor_found': True,
 'value': '456',
 'status': 'OCR_CANDIDATE',
 'method': 'LABEL_NUMBER',
 'evidence_text': 'দাগ নং: ৪৫৬'}

In [47]:
def extract_labeled_text(
    text,
    field_name,
    labels,
    max_length=120
):

    lines = get_lines(
        text
    )

    anchor_found = False

    for line in lines:

        for label in labels:

            if label not in line:
                continue

            anchor_found = True

            escaped_label = (
                re.escape(
                    label
                )
            )

            pattern = (
                escaped_label
                +
                r"\s*"
                r"[:ঃ\-–]\s*"
                r"(.+)$"
            )

            match = re.search(
                pattern,
                line
            )

            if not match:
                continue

            value = (
                match.group(1)
                .strip()
            )

            if (
                not value
                or len(value) > max_length
            ):
                continue

            return extraction_result(
                field_name=
                    field_name,

                anchor_found=
                    True,

                value=
                    value,

                method=
                    "LABEL_TEXT",

                evidence_text=
                    line,
            )

    return extraction_result(
        field_name=
            field_name,

        anchor_found=
            anchor_found,

        method=
            "LABEL_TEXT",
    )

In [48]:
AREA_UNITS = [
    "শতক",
    "ডেসিমেল",
    "একর",
    "কাঠা",
    "বিঘা",
    "ছটাক",
    "বর্গফুট",
]

In [49]:
def extract_labeled_area(
    text,
    field_name,
    labels
):

    lines = get_lines(
        text
    )

    anchor_found = False

    unit_pattern = "|".join(
        re.escape(unit)
        for unit in AREA_UNITS
    )

    for line in lines:

        normalized_line = (
            normalize_digits(
                line
            )
        )

        for label in labels:

            if label not in line:
                continue

            anchor_found = True

            escaped_label = (
                re.escape(
                    label
                )
            )

            pattern = (
                escaped_label
                +
                r".{0,30}?"
                r"([0-9]+(?:\.[0-9]+)?)"
                r"\s*"
                r"("
                + unit_pattern
                + r")"
            )

            match = re.search(
                pattern,
                normalized_line
            )

            if match:

                value = {
                    "amount":
                        match.group(1),

                    "unit":
                        match.group(2),
                }

                return extraction_result(
                    field_name=
                        field_name,

                    anchor_found=
                        True,

                    value=
                        value,

                    method=
                        "LABEL_AREA",

                    evidence_text=
                        line,
                )

    return extraction_result(
        field_name=
            field_name,

        anchor_found=
            anchor_found,

        method=
            "LABEL_AREA",
    )

In [50]:
area_test = """
হস্তান্তরিত জমির পরিমাণ: ৫.৫০ শতক
"""

extract_labeled_area(
    area_test,
    field_name="transferred_area",
    labels=[
        "হস্তান্তরিত জমির পরিমাণ",
        "জমির পরিমাণ",
    ]
)

{'field_name': 'transferred_area',
 'anchor_found': True,
 'value': {'amount': '5.50', 'unit': 'শতক'},
 'status': 'OCR_CANDIDATE',
 'method': 'LABEL_AREA',
 'evidence_text': 'হস্তান্তরিত জমির পরিমাণ: ৫.৫০ শতক'}

In [51]:
bad_area_test = """
জমির পরিমাণ: ৫৫০
"""

extract_labeled_area(
    bad_area_test,
    field_name="transferred_area",
    labels=[
        "জমির পরিমাণ"
    ]
)

{'field_name': 'transferred_area',
 'anchor_found': True,
 'value': None,
 'status': 'ANCHOR_ONLY',
 'method': 'LABEL_AREA',
 'evidence_text': None}

In [52]:
SURVEY_TYPES = [
    "CS",
    "SA",
    "RS",
    "BS",
    "BRS",
    "সি এস",
    "এস এ",
    "আর এস",
    "বি এস",
    "বি আর এস",
]

In [53]:
def extract_survey_type(
    text
):

    cleaned = clean_ocr_text(
        text
    )

    for survey in SURVEY_TYPES:

        pattern = (
            r"(?<!\S)"
            +
            re.escape(
                survey
            )
            +
            r"(?!\S)"
        )

        if re.search(
            pattern,
            cleaned,
            flags=re.IGNORECASE
        ):

            return extraction_result(
                field_name=
                    "survey_type",

                anchor_found=
                    True,

                value=
                    survey,

                method=
                    "EXPLICIT_SURVEY_TOKEN",
            )

    return extraction_result(
        field_name=
            "survey_type",

        anchor_found=
            False,

        method=
            "EXPLICIT_SURVEY_TOKEN",
    )

In [54]:
def extract_property_fields(
    page_texts
):

    page1 = page_texts.get(
        1,
        ""
    )

    page11 = page_texts.get(
        11,
        ""
    )

    page12 = page_texts.get(
        12,
        ""
    )

    schedule_text = "\n".join(
        [
            page11,
            page12,
        ]
    )

    results = {}

    results[
        "district"
    ] = extract_labeled_text(
        page1,
        "district",
        [
            "জেলা",
        ]
    )

    results[
        "upazila"
    ] = extract_labeled_text(
        page1,
        "upazila",
        [
            "উপজেলা",
            "থানা",
        ]
    )

    results[
        "mouza"
    ] = extract_labeled_text(
        schedule_text,
        "mouza",
        [
            "মৌজা",
        ]
    )

    results[
        "survey_type"
    ] = extract_survey_type(
        schedule_text
    )

    results[
        "khatian_no"
    ] = extract_labeled_number(
        schedule_text,
        "khatian_no",
        [
            "খতিয়ান",
            "খতিয়ান",
        ]
    )

    results[
        "dag_no"
    ] = extract_labeled_number(
        schedule_text,
        "dag_no",
        [
            "দাগ",
        ]
    )

    results[
        "total_property_area"
    ] = extract_labeled_area(
        schedule_text,
        "total_property_area",
        [
            "মোট জমির পরিমাণ",
            "মোট সম্পত্তির পরিমাণ",
        ]
    )

    results[
        "transferred_area"
    ] = extract_labeled_area(
        "\n".join(
            [
                page1,
                page11,
                page12,
            ]
        ),
        "transferred_area",
        [
            "হস্তান্তরিত জমির পরিমাণ",
            "হস্তান্তরিত সম্পত্তির পরিমাণ",
        ]
    )

    return results

In [55]:
blank_template_test_pages = {
    1: """
জেলা:
উপজেলা:
মৌজা:
হস্তান্তরিত জমির পরিমাণ:
""",

    11: """
সম্পত্তির তফসিল
মৌজা:
খতিয়ান নং:
দাগ নং:

প্রকাশ থাকে যে, অত্র দলিলের দাগ, খতিয়ান,
ইত্যাদি সম্পর্কে কোন ভুল থাকিলে...
""",

    12: """
জমির বিবরণ ও সীমানা
"""
}

In [56]:
blank_property_results = (
    extract_property_fields(
        blank_template_test_pages
    )
)

blank_property_results

{'district': {'field_name': 'district',
  'anchor_found': True,
  'value': None,
  'status': 'ANCHOR_ONLY',
  'method': 'LABEL_TEXT',
  'evidence_text': None},
 'upazila': {'field_name': 'upazila',
  'anchor_found': True,
  'value': None,
  'status': 'ANCHOR_ONLY',
  'method': 'LABEL_TEXT',
  'evidence_text': None},
 'mouza': {'field_name': 'mouza',
  'anchor_found': True,
  'value': None,
  'status': 'ANCHOR_ONLY',
  'method': 'LABEL_TEXT',
  'evidence_text': None},
 'survey_type': {'field_name': 'survey_type',
  'anchor_found': False,
  'value': None,
  'status': 'NOT_FOUND',
  'method': 'EXPLICIT_SURVEY_TOKEN',
  'evidence_text': None},
 'khatian_no': {'field_name': 'khatian_no',
  'anchor_found': True,
  'value': None,
  'status': 'ANCHOR_ONLY',
  'method': 'LABEL_NUMBER',
  'evidence_text': None},
 'dag_no': {'field_name': 'dag_no',
  'anchor_found': True,
  'value': None,
  'status': 'ANCHOR_ONLY',
  'method': 'LABEL_NUMBER',
  'evidence_text': None},
 'total_property_area': {'fi

In [57]:
blank_safety_df = pd.DataFrame(
    [
        {
            "field_name":
                field_name,

            "anchor_found":
                result[
                    "anchor_found"
                ],

            "value":
                result[
                    "value"
                ],

            "status":
                result[
                    "status"
                ],
        }

        for field_name, result
        in blank_property_results.items()
    ]
)

blank_safety_df

,field_name,anchor_found,value,status
0,district,True,None,ANCHOR_ONLY
1,upazila,True,None,ANCHOR_ONLY
2,mouza,True,None,ANCHOR_ONLY
3,survey_type,False,None,NOT_FOUND
4,khatian_no,True,None,ANCHOR_ONLY
5,dag_no,True,None,ANCHOR_ONLY
6,total_property_area,False,None,NOT_FOUND
7,transferred_area,True,None,ANCHOR_ONLY


In [58]:
false_value_count = sum(
    result[
        "value"
    ] is not None

    for result
    in blank_property_results.values()
)

print(
    "Values extracted from blank template:",
    false_value_count
)

Values extracted from blank template: 0


In [59]:
def apply_extraction_result(
    record,
    result,
    source_document,
    source_page=None,
    source_type="PRINTED_OCR",
):

    field_name = (
        result[
            "field_name"
        ]
    )

    field_record = (
        record[
            "fields"
        ][
            field_name
        ]
    )

    if (
        result[
            "status"
        ] == "NOT_FOUND"
    ):
        return

    field_record[
        "source_type"
    ] = source_type

    field_record[
        "source_document"
    ] = source_document

    field_record[
        "source_page"
    ] = source_page

    if (
        result[
            "status"
        ] == "ANCHOR_ONLY"
    ):

        field_record[
            "value"
        ] = None

        field_record[
            "verification_status"
        ] = "ANCHOR_ONLY"

        field_record[
            "review_required"
        ] = False

        return

    field_record[
        "value"
    ] = result[
        "value"
    ]

    field_record[
        "verification_status"
    ] = "OCR_CANDIDATE"

    field_record[
        "review_required"
    ] = True

In [60]:
step21c_template_record = deepcopy(
    saf_kabala_record
)

In [61]:
for result in (
    blank_property_results.values()
):

    apply_extraction_result(
        step21c_template_record,
        result,
        source_document=
            "সাফ-কবলা.pdf",

        source_type=
            "PRINTED_OCR",
    )

In [62]:
step21c_summary = (
    extraction_summary(
        step21c_template_record
    )
)

step21c_summary

{'document_id': 'downloaded_saf_kabala_template',
 'document_status': 'TEMPLATE_LIKE_UNFILLED',
 'total_fields': 19,
 'human_verified': 1,
 'ocr_candidates': 0,
 'not_found': 12,
 'review_required': 0,
 'transaction_engine_allowed': False}

In [63]:
assert (
    false_value_count
    == 0
)

assert (
    step21c_template_record[
        "document_status"
    ]
    ==
    "TEMPLATE_LIKE_UNFILLED"
)

assert (
    step21c_summary[
        "transaction_engine_allowed"
    ]
    is False
)

print(
    "Conservative extraction safety checks: PASS"
)

Conservative extraction safety checks: PASS


In [67]:
anchor_only_count = sum(
    result[
        "status"
    ]
    == "ANCHOR_ONLY"

    for result
    in blank_property_results.values()
)

not_found_count = sum(
    result[
        "status"
    ]
    == "NOT_FOUND"

    for result
    in blank_property_results.values()
)


print("STEP 21C SUMMARY")
print("----------------")

print(
    "Property fields tested:",
    len(
        blank_property_results
    )
)

print(
    "Anchor-only results:",
    anchor_only_count
)

print(
    "Not-found results:",
    not_found_count
)

print(
    "False values extracted:",
    false_value_count
)

print(
    "Template transaction allowed:",
    step21c_summary[
        "transaction_engine_allowed"
    ]
)

print(
    "Narrative khatian protection:",
    result is not None
)

print(
    "\nConservative Extraction Engine v1: READY"
)

STEP 21C SUMMARY
----------------
Property fields tested: 8
Anchor-only results: 6
Not-found results: 2
False values extracted: 0
Template transaction allowed: False
Narrative khatian protection: True

Conservative Extraction Engine v1: READY


In [65]:
narrative_protection_result = (
    extract_labeled_number(
        narrative_test,
        field_name="khatian_no",
        labels=[
            "খতিয়ান",
            "খতিয়ান",
        ]
    )
)

narrative_protection_pass = (
    narrative_protection_result[
        "value"
    ] is None
)

In [66]:
print(
    "Narrative khatian protection:",
    narrative_protection_pass
)

Narrative khatian protection: True


# STEP 21D — Positive Filled-Document Extraction Test

This step verifies that the conservative extractor can recover explicit
field values when a filled deed-like document actually contains them.

Only synthetic test evidence is used.

Goals:

- confirm explicit property values are extracted correctly
- confirm Bengali digits are normalized
- confirm all extracted values remain OCR candidates
- confirm unverified candidates cannot enter the risk engine
- keep synthetic test records blocked from transaction processing

In [68]:
synthetic_filled_pages = {
    1: """
জেলা: নমুনা জেলা
উপজেলা: নমুনা উপজেলা
হস্তান্তরিত জমির পরিমাণ: ৫.৫০ শতক
""",

    11: """
সম্পত্তির তফসিল

মৌজা: নমুনা মৌজা
আর এস

খতিয়ান নং: ১২৩
দাগ নং: ৪৫৬

মোট জমির পরিমাণ: ১০.৫০ শতক

প্রকাশ থাকে যে, অত্র দলিলের দাগ, খতিয়ান
ইত্যাদি সম্পর্কে কোন ভুল থাকিলে সংশোধনযোগ্য।
""",

    12: """
জমির বিবরণ ও সীমানা
"""
}

In [69]:
positive_property_results = (
    extract_property_fields(
        synthetic_filled_pages
    )
)

positive_property_results

{'district': {'field_name': 'district',
  'anchor_found': True,
  'value': 'নমুনা জেলা',
  'status': 'OCR_CANDIDATE',
  'method': 'LABEL_TEXT',
  'evidence_text': 'জেলা: নমুনা জেলা'},
 'upazila': {'field_name': 'upazila',
  'anchor_found': True,
  'value': 'নমুনা উপজেলা',
  'status': 'OCR_CANDIDATE',
  'method': 'LABEL_TEXT',
  'evidence_text': 'উপজেলা: নমুনা উপজেলা'},
 'mouza': {'field_name': 'mouza',
  'anchor_found': True,
  'value': 'নমুনা মৌজা',
  'status': 'OCR_CANDIDATE',
  'method': 'LABEL_TEXT',
  'evidence_text': 'মৌজা: নমুনা মৌজা'},
 'survey_type': {'field_name': 'survey_type',
  'anchor_found': True,
  'value': 'আর এস',
  'status': 'OCR_CANDIDATE',
  'method': 'EXPLICIT_SURVEY_TOKEN',
  'evidence_text': None},
 'khatian_no': {'field_name': 'khatian_no',
  'anchor_found': True,
  'value': '123',
  'status': 'OCR_CANDIDATE',
  'method': 'LABEL_NUMBER',
  'evidence_text': 'খতিয়ান নং: ১২৩'},
 'dag_no': {'field_name': 'dag_no',
  'anchor_found': True,
  'value': '456',
  'statu

In [70]:
positive_extraction_df = pd.DataFrame(
    [
        {
            "field_name":
                field_name,

            "anchor_found":
                result[
                    "anchor_found"
                ],

            "value":
                result[
                    "value"
                ],

            "status":
                result[
                    "status"
                ],

            "method":
                result[
                    "method"
                ],
        }

        for field_name, result
        in positive_property_results.items()
    ]
)

positive_extraction_df

,field_name,anchor_found,value,status,method
0,district,True,নমুনা জেলা,OCR_CANDIDATE,LABEL_TEXT
1,upazila,True,নমুনা উপজেলা,OCR_CANDIDATE,LABEL_TEXT
2,mouza,True,নমুনা মৌজা,OCR_CANDIDATE,LABEL_TEXT
3,survey_type,True,আর এস,OCR_CANDIDATE,EXPLICIT_SURVEY_TOKEN
4,khatian_no,True,123,OCR_CANDIDATE,LABEL_NUMBER
5,dag_no,True,456,OCR_CANDIDATE,LABEL_NUMBER
6,total_property_area,True,"{'amount': '10.50', 'unit': 'শতক'}",OCR_CANDIDATE,LABEL_AREA
7,transferred_area,True,"{'amount': '5.50', 'unit': 'শতক'}",OCR_CANDIDATE,LABEL_AREA


In [71]:
expected_property_values = {
    "district":
        "নমুনা জেলা",

    "upazila":
        "নমুনা উপজেলা",

    "mouza":
        "নমুনা মৌজা",

    "survey_type":
        "আর এস",

    "khatian_no":
        "123",

    "dag_no":
        "456",

    "total_property_area":
        {
            "amount": "10.50",
            "unit": "শতক",
        },

    "transferred_area":
        {
            "amount": "5.50",
            "unit": "শতক",
        },
}

In [72]:
positive_test_rows = []

for field_name, expected_value in (
    expected_property_values.items()
):

    actual_value = (
        positive_property_results[
            field_name
        ][
            "value"
        ]
    )

    positive_test_rows.append({
        "field_name":
            field_name,

        "expected_value":
            expected_value,

        "actual_value":
            actual_value,

        "match":
            actual_value
            == expected_value,
    })


positive_test_df = pd.DataFrame(
    positive_test_rows
)

positive_test_df

,field_name,expected_value,actual_value,match
0,district,নমুনা জেলা,নমুনা জেলা,True
1,upazila,নমুনা উপজেলা,নমুনা উপজেলা,True
2,mouza,নমুনা মৌজা,নমুনা মৌজা,True
3,survey_type,আর এস,আর এস,True
4,khatian_no,123,123,True
5,dag_no,456,456,True
6,total_property_area,"{'amount': '10.50', 'unit': 'শতক'}","{'amount': '10.50', 'unit': 'শতক'}",True
7,transferred_area,"{'amount': '5.50', 'unit': 'শতক'}","{'amount': '5.50', 'unit': 'শতক'}",True


In [73]:
exact_match_count = int(
    positive_test_df[
        "match"
    ].sum()
)

print(
    "Exact matches:",
    exact_match_count,
    "/",
    len(
        positive_test_df
    )
)

Exact matches: 8 / 8


In [74]:
ocr_candidate_count = sum(
    result[
        "status"
    ]
    == "OCR_CANDIDATE"

    for result
    in positive_property_results.values()
)

print(
    "OCR candidates:",
    ocr_candidate_count
)

OCR candidates: 8


In [75]:
synthetic_test_record = (
    create_empty_extraction_record(
        "synthetic_filled_property_test"
    )
)

synthetic_test_record[
    "document_status"
] = (
    "SYNTHETIC_TEST_ONLY"
)

In [76]:
def document_allowed_for_transaction_engine(
    record
):

    blocked_statuses = {
        "TEMPLATE_LIKE_UNFILLED",
        "NON_DEED",
        "OCR_UNVERIFIED",
        "SYNTHETIC_TEST_ONLY",
    }

    return (
        record[
            "document_status"
        ]
        not in blocked_statuses
    )

In [77]:
print(
    "Synthetic test allowed:",
    document_allowed_for_transaction_engine(
        synthetic_test_record
    )
)

Synthetic test allowed: False


In [78]:
for result in (
    positive_property_results.values()
):

    apply_extraction_result(
        synthetic_test_record,
        result,
        source_document=
            "SYNTHETIC_TEST_TEXT",

        source_type=
            "PRINTED_OCR",
    )

In [79]:
synthetic_test_summary = (
    extraction_summary(
        synthetic_test_record
    )
)

synthetic_test_summary

{'document_id': 'synthetic_filled_property_test',
 'document_status': 'SYNTHETIC_TEST_ONLY',
 'total_fields': 19,
 'human_verified': 0,
 'ocr_candidates': 8,
 'not_found': 11,
 'review_required': 8,
 'transaction_engine_allowed': False}

In [80]:
candidate_gate_results = []

for field_name in (
    expected_property_values.keys()
):

    field_record = (
        synthetic_test_record[
            "fields"
        ][
            field_name
        ]
    )

    candidate_gate_results.append({
        "field_name":
            field_name,

        "verification_status":
            field_record[
                "verification_status"
            ],

        "risk_engine_allowed":
            field_allowed_for_risk_engine(
                field_record
            ),
    })


candidate_gate_df = pd.DataFrame(
    candidate_gate_results
)

candidate_gate_df

,field_name,verification_status,risk_engine_allowed
0,district,OCR_CANDIDATE,False
1,upazila,OCR_CANDIDATE,False
2,mouza,OCR_CANDIDATE,False
3,survey_type,OCR_CANDIDATE,False
4,khatian_no,OCR_CANDIDATE,False
5,dag_no,OCR_CANDIDATE,False
6,total_property_area,OCR_CANDIDATE,False
7,transferred_area,OCR_CANDIDATE,False


In [81]:
blocked_candidate_count = int(
    (
        candidate_gate_df[
            "risk_engine_allowed"
        ] == False
    ).sum()
)

print(
    "Unverified candidates blocked:",
    blocked_candidate_count,
    "/",
    len(
        candidate_gate_df
    )
)

Unverified candidates blocked: 8 / 8


In [82]:
assert (
    exact_match_count
    == 8
)

assert (
    ocr_candidate_count
    == 8
)

assert (
    blocked_candidate_count
    == 8
)

assert (
    synthetic_test_summary[
        "transaction_engine_allowed"
    ]
    is False
)

assert (
    positive_property_results[
        "khatian_no"
    ][
        "value"
    ]
    == "123"
)

assert (
    positive_property_results[
        "dag_no"
    ][
        "value"
    ]
    == "456"
)

print(
    "Positive extraction safety tests: PASS"
)

Positive extraction safety tests: PASS


In [83]:
print("STEP 21D SUMMARY")
print("----------------")

print(
    "Property fields tested:",
    len(
        expected_property_values
    )
)

print(
    "Explicit values extracted:",
    sum(
        result[
            "value"
        ] is not None

        for result
        in positive_property_results.values()
    )
)

print(
    "Exact expected matches:",
    exact_match_count,
    "/",
    len(
        expected_property_values
    )
)

print(
    "OCR candidate fields:",
    ocr_candidate_count
)

print(
    "Unverified candidates blocked:",
    blocked_candidate_count,
    "/",
    len(
        candidate_gate_df
    )
)

print(
    "Synthetic test transaction allowed:",
    synthetic_test_summary[
        "transaction_engine_allowed"
    ]
)

print(
    "\nPositive Extraction Test: PASS"
)

STEP 21D SUMMARY
----------------
Property fields tested: 8
Explicit values extracted: 8
Exact expected matches: 8 / 8
OCR candidate fields: 8
Unverified candidates blocked: 8 / 8
Synthetic test transaction allowed: False

Positive Extraction Test: PASS


# STEP 21E — Human Verification and ParcelKey Eligibility

This step verifies the transition:

OCR candidate
→ human review
→ canonical normalization
→ verified structured field
→ derived ParcelKey

Rules:

- OCR candidates cannot create a ParcelKey.
- All ParcelKey source fields must be human verified.
- The original OCR candidate is preserved for provenance.
- ParcelKey is derived only from verified source fields.
- Synthetic test records remain blocked from transaction processing.

In [84]:
PARCEL_KEY_FIELDS = [
    "district",
    "upazila",
    "mouza",
    "survey_type",
    "khatian_no",
    "dag_no",
]

print(
    "ParcelKey source fields:",
    len(PARCEL_KEY_FIELDS)
)

ParcelKey source fields: 6


In [85]:
def normalize_text_value(value):

    if value is None:
        return None

    return " ".join(
        str(value)
        .strip()
        .split()
    )

In [86]:
SURVEY_ALIAS_MAP = {
    "CS": "CS",
    "সি এস": "CS",

    "SA": "SA",
    "এস এ": "SA",

    "RS": "RS",
    "আর এস": "RS",

    "BS": "BS",
    "বি এস": "BS",

    "BRS": "BRS",
    "বি আর এস": "BRS",
}

In [87]:
def normalize_survey_value(value):

    cleaned = normalize_text_value(
        value
    )

    if cleaned is None:
        return None

    return SURVEY_ALIAS_MAP.get(
        cleaned.upper(),
        SURVEY_ALIAS_MAP.get(
            cleaned,
            cleaned
        )
    )

In [88]:
def normalize_simple_land_number(
    value
):

    if value is None:
        return None

    normalized = normalize_digits(
        str(value)
    )

    digits = re.sub(
        r"\D",
        "",
        normalized
    )

    return (
        digits
        if digits
        else None
    )

In [89]:
def normalize_verified_value(
    field_name,
    value
):

    if field_name in {
        "district",
        "upazila",
        "mouza",
    }:

        return normalize_text_value(
            value
        )

    if field_name == "survey_type":

        return normalize_survey_value(
            value
        )

    if field_name in {
        "khatian_no",
        "dag_no",
    }:

        return normalize_simple_land_number(
            value
        )

    return value

In [90]:
print(
    normalize_verified_value(
        "survey_type",
        "আর এস"
    )
)

print(
    normalize_verified_value(
        "khatian_no",
        "১২৩"
    )
)

RS
123


In [91]:
def human_verify_candidate(
    record,
    field_name,
    verified_value=None,
    verified_by="DEMO_REVIEWER",
    notes=None,
):

    field_record = (
        record[
            "fields"
        ][
            field_name
        ]
    )

    original_candidate = deepcopy(
        field_record[
            "value"
        ]
    )

    if verified_value is None:

        verified_value = (
            original_candidate
        )

    normalized_value = (
        normalize_verified_value(
            field_name,
            verified_value
        )
    )

    field_record[
        "candidate_value"
    ] = original_candidate

    field_record[
        "value"
    ] = normalized_value

    field_record[
        "verification_status"
    ] = "HUMAN_VERIFIED"

    field_record[
        "review_required"
    ] = False

    field_record[
        "verified_by"
    ] = verified_by

    field_record[
        "verification_notes"
    ] = notes

    return field_record

In [92]:
def parcel_key_eligibility(
    record
):

    blocked_fields = []

    for field_name in (
        PARCEL_KEY_FIELDS
    ):

        field_record = (
            record[
                "fields"
            ][
                field_name
            ]
        )

        if (
            not field_allowed_for_risk_engine(
                field_record
            )
        ):

            blocked_fields.append(
                field_name
            )

            continue

        if (
            field_record[
                "value"
            ]
            is None
        ):

            blocked_fields.append(
                field_name
            )

    return {
        "eligible":
            len(
                blocked_fields
            ) == 0,

        "blocked_fields":
            blocked_fields,
    }

In [93]:
before_verification = (
    parcel_key_eligibility(
        synthetic_test_record
    )
)

before_verification

{'eligible': False,
 'blocked_fields': ['district',
  'upazila',
  'mouza',
  'survey_type',
  'khatian_no',
  'dag_no']}

In [94]:
print(
    "Eligible before verification:",
    before_verification[
        "eligible"
    ]
)

print(
    "Blocked fields:",
    len(
        before_verification[
            "blocked_fields"
        ]
    )
)

Eligible before verification: False
Blocked fields: 6


In [95]:
verified_test_record = deepcopy(
    synthetic_test_record
)

In [96]:
for field_name in (
    PARCEL_KEY_FIELDS
):

    human_verify_candidate(
        verified_test_record,
        field_name,
        verified_by=
            "DEMO_REVIEWER",

        notes=
            "Verified against synthetic test evidence."
    )

In [97]:
verified_parcel_rows = []

for field_name in (
    PARCEL_KEY_FIELDS
):

    field_record = (
        verified_test_record[
            "fields"
        ][
            field_name
        ]
    )

    verified_parcel_rows.append({
        "field_name":
            field_name,

        "candidate_value":
            field_record.get(
                "candidate_value"
            ),

        "canonical_value":
            field_record[
                "value"
            ],

        "verification_status":
            field_record[
                "verification_status"
            ],

        "risk_engine_allowed":
            field_allowed_for_risk_engine(
                field_record
            ),
    })


verified_parcel_df = pd.DataFrame(
    verified_parcel_rows
)

verified_parcel_df

,field_name,candidate_value,canonical_value,verification_status,risk_engine_allowed
0,district,নমুনা জেলা,নমুনা জেলা,HUMAN_VERIFIED,True
1,upazila,নমুনা উপজেলা,নমুনা উপজেলা,HUMAN_VERIFIED,True
2,mouza,নমুনা মৌজা,নমুনা মৌজা,HUMAN_VERIFIED,True
3,survey_type,আর এস,RS,HUMAN_VERIFIED,True
4,khatian_no,123,123,HUMAN_VERIFIED,True
5,dag_no,456,456,HUMAN_VERIFIED,True


In [98]:
after_verification = (
    parcel_key_eligibility(
        verified_test_record
    )
)

after_verification

{'eligible': True, 'blocked_fields': []}

In [99]:
def derive_verified_parcel_key(
    record
):

    eligibility = (
        parcel_key_eligibility(
            record
        )
    )

    if not eligibility[
        "eligible"
    ]:

        return None

    values = [
        str(
            record[
                "fields"
            ][
                field_name
            ][
                "value"
            ]
        )

        for field_name
        in PARCEL_KEY_FIELDS
    ]

    return "|".join(
        values
    )

In [100]:
parcel_key = (
    derive_verified_parcel_key(
        verified_test_record
    )
)

print(
    "ParcelKey:",
    parcel_key
)

ParcelKey: নমুনা জেলা|নমুনা উপজেলা|নমুনা মৌজা|RS|123|456


In [101]:
verified_test_record[
    "derived_fields"
] = {
    "parcel_key": {
        "value":
            parcel_key,

        "verification_status":
            "DERIVED_FROM_VERIFIED_FIELDS",

        "source_fields":
            PARCEL_KEY_FIELDS.copy(),

        "review_required":
            False,
    },

    "portion_id": {
        "value":
            None,

        "verification_status":
            "NOT_DERIVED",

        "review_required":
            True,

        "notes":
            (
                "Requires schedule, boundary, "
                "and/or share evidence."
            ),
    },
}

In [102]:
parcel_key_record = (
    verified_test_record[
        "derived_fields"
    ][
        "parcel_key"
    ]
)

print(
    "ParcelKey status:",
    parcel_key_record[
        "verification_status"
    ]
)

print(
    "ParcelKey allowed:",
    field_allowed_for_risk_engine(
        parcel_key_record
    )
)

ParcelKey status: DERIVED_FROM_VERIFIED_FIELDS
ParcelKey allowed: True


In [103]:
print(
    "Portion ID:",
    verified_test_record[
        "derived_fields"
    ][
        "portion_id"
    ][
        "value"
    ]
)

Portion ID: None


In [104]:
remaining_candidates = [
    field_name

    for field_name, field_record
    in verified_test_record[
        "fields"
    ].items()

    if (
        field_record[
            "verification_status"
        ]
        == "OCR_CANDIDATE"
    )
]

remaining_candidates

['total_property_area', 'transferred_area']

In [105]:
print(
    "Synthetic document transaction allowed:",
    document_allowed_for_transaction_engine(
        verified_test_record
    )
)

Synthetic document transaction allowed: False


In [106]:
assert (
    before_verification[
        "eligible"
    ]
    is False
)

assert (
    len(
        before_verification[
            "blocked_fields"
        ]
    )
    == 6
)

assert (
    after_verification[
        "eligible"
    ]
    is True
)

assert (
    parcel_key
    is not None
)

assert (
    parcel_key_record[
        "verification_status"
    ]
    ==
    "DERIVED_FROM_VERIFIED_FIELDS"
)

assert (
    field_allowed_for_risk_engine(
        parcel_key_record
    )
    is True
)

assert (
    verified_test_record[
        "derived_fields"
    ][
        "portion_id"
    ][
        "value"
    ]
    is None
)

assert (
    document_allowed_for_transaction_engine(
        verified_test_record
    )
    is False
)

print(
    "Human verification and ParcelKey tests: PASS"
)

Human verification and ParcelKey tests: PASS


In [107]:
human_verified_parcel_fields = sum(
    verified_test_record[
        "fields"
    ][
        field_name
    ][
        "verification_status"
    ]
    == "HUMAN_VERIFIED"

    for field_name
    in PARCEL_KEY_FIELDS
)


print("STEP 21E SUMMARY")
print("----------------")

print(
    "ParcelKey source fields:",
    len(
        PARCEL_KEY_FIELDS
    )
)

print(
    "Eligible before verification:",
    before_verification[
        "eligible"
    ]
)

print(
    "Blocked before verification:",
    len(
        before_verification[
            "blocked_fields"
        ]
    )
)

print(
    "Human-verified parcel fields:",
    human_verified_parcel_fields
)

print(
    "Eligible after verification:",
    after_verification[
        "eligible"
    ]
)

print(
    "ParcelKey derived:",
    parcel_key is not None
)

print(
    "ParcelKey status:",
    parcel_key_record[
        "verification_status"
    ]
)

print(
    "Remaining OCR candidates:",
    len(
        remaining_candidates
    )
)

print(
    "Portion ID derived:",
    verified_test_record[
        "derived_fields"
    ][
        "portion_id"
    ][
        "value"
    ]
    is not None
)

print(
    "Synthetic transaction allowed:",
    document_allowed_for_transaction_engine(
        verified_test_record
    )
)

print(
    "\nHuman Verification + ParcelKey Gate: PASS"
)

STEP 21E SUMMARY
----------------
ParcelKey source fields: 6
Eligible before verification: False
Blocked before verification: 6
Human-verified parcel fields: 6
Eligible after verification: True
ParcelKey derived: True
ParcelKey status: DERIVED_FROM_VERIFIED_FIELDS
Remaining OCR candidates: 2
Portion ID derived: False
Synthetic transaction allowed: False

Human Verification + ParcelKey Gate: PASS


# STEP 21F — Rule-Level Risk Readiness Gate

This step determines which BhumiGuard risk checks are allowed to run
from a structured extraction record.

Each rule has its own minimum verified evidence requirements.

Important principles:

- A partially extracted deed must not trigger unsupported risk rules.
- OCR candidates do not count as verified evidence.
- ParcelKey must come only from verified parcel fields.
- Double-sale checking requires portion-level evidence.
- Mutation mismatch requires verified buyer and parcel identity.
- Rule readiness does not override document-level blocking.

In [108]:
def verified_field_available(
    record,
    field_name
):

    field_record = (
        record[
            "fields"
        ][
            field_name
        ]
    )

    return (
        field_record[
            "value"
        ] is not None
        and
        field_allowed_for_risk_engine(
            field_record
        )
    )

In [109]:
def verified_derived_field_available(
    record,
    field_name
):

    derived_fields = (
        record.get(
            "derived_fields",
            {}
        )
    )

    if field_name not in derived_fields:
        return False

    field_record = (
        derived_fields[
            field_name
        ]
    )

    return (
        field_record.get(
            "value"
        )
        is not None
        and
        field_record.get(
            "verification_status"
        )
        ==
        "DERIVED_FROM_VERIFIED_FIELDS"
    )

In [110]:
print(
    "ParcelKey verified:",
    verified_derived_field_available(
        verified_test_record,
        "parcel_key"
    )
)

print(
    "Portion ID verified:",
    verified_derived_field_available(
        verified_test_record,
        "portion_id"
    )
)

ParcelKey verified: True
Portion ID verified: False


In [111]:
RISK_RULE_REQUIREMENTS = {

    "DUPLICATE_DEED": {
        "fields": [
            "deed_id",
        ],
        "derived_fields": [],
    },

    "SELLER_NOT_OWNER": {
        "fields": [
            "seller_id",
        ],
        "derived_fields": [
            "parcel_key",
        ],
    },

    "OVER_TRANSFER": {
        "fields": [
            "seller_id",
            "transferred_area",
        ],
        "derived_fields": [
            "parcel_key",
        ],
    },

    "DOUBLE_SALE": {
        "fields": [
            "seller_id",
        ],
        "derived_fields": [
            "parcel_key",
            "portion_id",
        ],
    },

    "MUTATION_MISMATCH": {
        "fields": [
            "buyer_id",
        ],
        "derived_fields": [
            "parcel_key",
        ],
    },
}

In [112]:
def evaluate_rule_readiness(
    record,
    rule_name
):

    requirements = (
        RISK_RULE_REQUIREMENTS[
            rule_name
        ]
    )

    missing_fields = []

    missing_derived_fields = []

    for field_name in (
        requirements[
            "fields"
        ]
    ):

        if not verified_field_available(
            record,
            field_name
        ):

            missing_fields.append(
                field_name
            )

    for field_name in (
        requirements[
            "derived_fields"
        ]
    ):

        if not verified_derived_field_available(
            record,
            field_name
        ):

            missing_derived_fields.append(
                field_name
            )

    ready = (
        len(
            missing_fields
        ) == 0
        and
        len(
            missing_derived_fields
        ) == 0
    )

    return {
        "rule":
            rule_name,

        "ready":
            ready,

        "missing_fields":
            missing_fields,

        "missing_derived_fields":
            missing_derived_fields,
    }

In [113]:
evaluate_rule_readiness(
    verified_test_record,
    "OVER_TRANSFER"
)

{'rule': 'OVER_TRANSFER',
 'ready': False,
 'missing_fields': ['seller_id', 'transferred_area'],
 'missing_derived_fields': []}

In [114]:
rule_readiness_results = []

for rule_name in (
    RISK_RULE_REQUIREMENTS
):

    result = (
        evaluate_rule_readiness(
            verified_test_record,
            rule_name
        )
    )

    rule_readiness_results.append(
        result
    )


rule_readiness_df = pd.DataFrame(
    rule_readiness_results
)

rule_readiness_df

,rule,ready,missing_fields,missing_derived_fields
0,DUPLICATE_DEED,False,[deed_id],[]
1,SELLER_NOT_OWNER,False,[seller_id],[]
2,OVER_TRANSFER,False,"[seller_id, transferred_area]",[]
3,DOUBLE_SALE,False,[seller_id],[portion_id]
4,MUTATION_MISMATCH,False,[buyer_id],[]


In [115]:
rule_readiness_display_df = (
    rule_readiness_df[
        [
            "rule",
            "ready",
            "missing_fields",
            "missing_derived_fields",
        ]
    ]
)

rule_readiness_display_df

,rule,ready,missing_fields,missing_derived_fields
0,DUPLICATE_DEED,False,[deed_id],[]
1,SELLER_NOT_OWNER,False,[seller_id],[]
2,OVER_TRANSFER,False,"[seller_id, transferred_area]",[]
3,DOUBLE_SALE,False,[seller_id],[portion_id]
4,MUTATION_MISMATCH,False,[buyer_id],[]


In [116]:
ready_rule_count = int(
    rule_readiness_df[
        "ready"
    ].sum()
)

blocked_rule_count = (
    len(
        rule_readiness_df
    )
    - ready_rule_count
)

print(
    "Ready rules:",
    ready_rule_count
)

print(
    "Blocked rules:",
    blocked_rule_count
)

Ready rules: 0
Blocked rules: 5


In [117]:
TRANSACTION_CORE_FIELDS = [
    "deed_id",
    "execution_date",
    "seller_id",
    "buyer_id",
    "transferred_area",
]

In [118]:
def transaction_core_readiness(
    record
):

    missing_fields = []

    for field_name in (
        TRANSACTION_CORE_FIELDS
    ):

        if not verified_field_available(
            record,
            field_name
        ):

            missing_fields.append(
                field_name
            )

    parcel_key_ready = (
        verified_derived_field_available(
            record,
            "parcel_key"
        )
    )

    ready = (
        len(
            missing_fields
        ) == 0
        and
        parcel_key_ready
    )

    return {
        "ready":
            ready,

        "missing_fields":
            missing_fields,

        "parcel_key_ready":
            parcel_key_ready,
    }

In [119]:
transaction_readiness = (
    transaction_core_readiness(
        verified_test_record
    )
)

transaction_readiness

{'ready': False,
 'missing_fields': ['deed_id',
  'execution_date',
  'seller_id',
  'buyer_id',
  'transferred_area'],
 'parcel_key_ready': True}

In [120]:
def final_transaction_gate(
    record
):

    completeness = (
        transaction_core_readiness(
            record
        )
    )

    document_allowed = (
        document_allowed_for_transaction_engine(
            record
        )
    )

    return {
        "core_ready":
            completeness[
                "ready"
            ],

        "document_allowed":
            document_allowed,

        "final_allowed":
            (
                completeness[
                    "ready"
                ]
                and
                document_allowed
            ),

        "missing_fields":
            completeness[
                "missing_fields"
            ],

        "parcel_key_ready":
            completeness[
                "parcel_key_ready"
            ],
    }

In [121]:
final_gate_result = (
    final_transaction_gate(
        verified_test_record
    )
)

final_gate_result

{'core_ready': False,
 'document_allowed': False,
 'final_allowed': False,
 'missing_fields': ['deed_id',
  'execution_date',
  'seller_id',
  'buyer_id',
  'transferred_area'],
 'parcel_key_ready': True}

In [122]:
def rule_execution_allowed(
    record,
    rule_name
):

    readiness = (
        evaluate_rule_readiness(
            record,
            rule_name
        )
    )

    document_allowed = (
        document_allowed_for_transaction_engine(
            record
        )
    )

    return (
        readiness[
            "ready"
        ]
        and
        document_allowed
    )

In [123]:
rule_execution_rows = []

for rule_name in (
    RISK_RULE_REQUIREMENTS
):

    rule_execution_rows.append({
        "rule":
            rule_name,

        "evidence_ready":
            evaluate_rule_readiness(
                verified_test_record,
                rule_name
            )[
                "ready"
            ],

        "execution_allowed":
            rule_execution_allowed(
                verified_test_record,
                rule_name
            ),
    })


rule_execution_df = pd.DataFrame(
    rule_execution_rows
)

rule_execution_df

,rule,evidence_ready,execution_allowed
0,DUPLICATE_DEED,False,False
1,SELLER_NOT_OWNER,False,False
2,OVER_TRANSFER,False,False
3,DOUBLE_SALE,False,False
4,MUTATION_MISMATCH,False,False


In [124]:
double_sale_readiness = (
    evaluate_rule_readiness(
        verified_test_record,
        "DOUBLE_SALE"
    )
)

print(
    "Double-sale ready:",
    double_sale_readiness[
        "ready"
    ]
)

print(
    "Missing derived evidence:",
    double_sale_readiness[
        "missing_derived_fields"
    ]
)

Double-sale ready: False
Missing derived evidence: ['portion_id']


In [125]:
def rule_readiness_status(
    readiness_result
):

    if readiness_result[
        "ready"
    ]:
        return "READY"

    return "INSUFFICIENT_VERIFIED_EVIDENCE"

In [126]:
rule_readiness_df[
    "readiness_status"
] = (
    rule_readiness_df.apply(
        lambda row:
            rule_readiness_status(
                row
            ),
        axis=1
    )
)

In [127]:
assert (
    ready_rule_count
    == 0
)

assert (
    blocked_rule_count
    == 5
)

assert (
    transaction_readiness[
        "ready"
    ]
    is False
)

assert (
    transaction_readiness[
        "parcel_key_ready"
    ]
    is True
)

assert (
    final_gate_result[
        "final_allowed"
    ]
    is False
)

assert (
    double_sale_readiness[
        "ready"
    ]
    is False
)

assert (
    "portion_id"
    in double_sale_readiness[
        "missing_derived_fields"
    ]
)

assert (
    rule_execution_df[
        "execution_allowed"
    ].sum()
    == 0
)

print(
    "Risk readiness safety checks: PASS"
)

Risk readiness safety checks: PASS


In [128]:
print("STEP 21F SUMMARY")
print("----------------")

print(
    "Risk rules defined:",
    len(
        RISK_RULE_REQUIREMENTS
    )
)

print(
    "Ready risk rules:",
    ready_rule_count
)

print(
    "Blocked risk rules:",
    blocked_rule_count
)

print(
    "ParcelKey available:",
    verified_derived_field_available(
        verified_test_record,
        "parcel_key"
    )
)

print(
    "Portion ID available:",
    verified_derived_field_available(
        verified_test_record,
        "portion_id"
    )
)

print(
    "Transaction core ready:",
    transaction_readiness[
        "ready"
    ]
)

print(
    "Document-level permission:",
    final_gate_result[
        "document_allowed"
    ]
)

print(
    "Final transaction allowed:",
    final_gate_result[
        "final_allowed"
    ]
)

print(
    "Rules executed without evidence:",
    int(
        rule_execution_df[
            "execution_allowed"
        ].sum()
    )
)

print(
    "\nRule-Level Readiness Gate: PASS"
)

STEP 21F SUMMARY
----------------
Risk rules defined: 5
Ready risk rules: 0
Blocked risk rules: 5
ParcelKey available: True
Portion ID available: False
Transaction core ready: False
Document-level permission: False
Final transaction allowed: False
Rules executed without evidence: 0

Rule-Level Readiness Gate: PASS


# STEP 21G — Fully Verified Synthetic Transaction Readiness Test

This step creates a fully human-verified synthetic transaction record
for readiness testing.

Goals:

- complete the minimum transaction-core evidence
- verify OCR candidates through the human-review gate
- confirm supported risk rules become READY
- confirm DOUBLE_SALE remains blocked without portion-level evidence
- keep the synthetic record blocked from live transaction execution

No real person or real land information is used.

In [129]:
step21g_record = deepcopy(
    verified_test_record
)

print(
    "Document status:",
    step21g_record[
        "document_status"
    ]
)

Document status: SYNTHETIC_TEST_ONLY


In [130]:
synthetic_transaction_candidates = {
    "deed_id":
        "DEED-SYN-001",

    "execution_date":
        "2026-10-16",

    "seller_id":
        "SELLER-SYN-001",

    "buyer_id":
        "BUYER-SYN-001",
}

In [131]:
for field_name, value in (
    synthetic_transaction_candidates.items()
):

    add_field_candidate(
        step21g_record,
        field_name=field_name,
        value=value,
        source_type="PRINTED_OCR",
        source_document="SYNTHETIC_TEST_TEXT",
        notes="Synthetic candidate for readiness testing."
    )

In [132]:
candidate_status_rows = []

for field_name in (
    synthetic_transaction_candidates
):

    field_record = (
        step21g_record[
            "fields"
        ][
            field_name
        ]
    )

    candidate_status_rows.append({
        "field_name":
            field_name,

        "value":
            field_record[
                "value"
            ],

        "verification_status":
            field_record[
                "verification_status"
            ],

        "risk_engine_allowed":
            field_allowed_for_risk_engine(
                field_record
            ),
    })


step21g_candidates_df = pd.DataFrame(
    candidate_status_rows
)

step21g_candidates_df

,field_name,value,verification_status,risk_engine_allowed
0,deed_id,DEED-SYN-001,OCR_CANDIDATE,False
1,execution_date,2026-10-16,OCR_CANDIDATE,False
2,seller_id,SELLER-SYN-001,OCR_CANDIDATE,False
3,buyer_id,BUYER-SYN-001,OCR_CANDIDATE,False


In [133]:
step21g_record[
    "fields"
][
    "transferred_area"
]

{'field_name': 'transferred_area',
 'value': {'amount': '5.50', 'unit': 'শতক'},
 'source_type': 'PRINTED_OCR',
 'source_document': 'SYNTHETIC_TEST_TEXT',
 'source_page': None,
 'ocr_confidence': None,
 'verification_status': 'OCR_CANDIDATE',
 'review_required': True,
 'notes': None}

In [134]:
fields_to_verify = [
    "deed_id",
    "execution_date",
    "seller_id",
    "buyer_id",
    "transferred_area",
]

In [135]:
for field_name in (
    fields_to_verify
):

    human_verify_candidate(
        step21g_record,
        field_name,
        verified_by=
            "DEMO_REVIEWER",

        notes=
            "Human verified synthetic readiness test value."
    )

In [136]:
verified_transaction_rows = []

for field_name in (
    fields_to_verify
):

    field_record = (
        step21g_record[
            "fields"
        ][
            field_name
        ]
    )

    verified_transaction_rows.append({
        "field_name":
            field_name,

        "value":
            field_record[
                "value"
            ],

        "verification_status":
            field_record[
                "verification_status"
            ],

        "risk_engine_allowed":
            field_allowed_for_risk_engine(
                field_record
            ),
    })


verified_transaction_df = pd.DataFrame(
    verified_transaction_rows
)

verified_transaction_df

,field_name,value,verification_status,risk_engine_allowed
0,deed_id,DEED-SYN-001,HUMAN_VERIFIED,True
1,execution_date,2026-10-16,HUMAN_VERIFIED,True
2,seller_id,SELLER-SYN-001,HUMAN_VERIFIED,True
3,buyer_id,BUYER-SYN-001,HUMAN_VERIFIED,True
4,transferred_area,"{'amount': '5.50', 'unit': 'শতক'}",HUMAN_VERIFIED,True


In [137]:
step21g_parcel_key = (
    step21g_record[
        "derived_fields"
    ][
        "parcel_key"
    ][
        "value"
    ]
)

print(
    "ParcelKey:",
    step21g_parcel_key
)

ParcelKey: নমুনা জেলা|নমুনা উপজেলা|নমুনা মৌজা|RS|123|456


In [138]:
step21g_portion_id = (
    step21g_record[
        "derived_fields"
    ][
        "portion_id"
    ][
        "value"
    ]
)

print(
    "Portion ID:",
    step21g_portion_id
)

Portion ID: None


In [139]:
step21g_transaction_readiness = (
    transaction_core_readiness(
        step21g_record
    )
)

step21g_transaction_readiness

{'ready': True, 'missing_fields': [], 'parcel_key_ready': True}

In [140]:
step21g_rule_results = []

for rule_name in (
    RISK_RULE_REQUIREMENTS
):

    result = (
        evaluate_rule_readiness(
            step21g_record,
            rule_name
        )
    )

    step21g_rule_results.append(
        result
    )


step21g_rule_df = pd.DataFrame(
    step21g_rule_results
)

step21g_rule_df

,rule,ready,missing_fields,missing_derived_fields
0,DUPLICATE_DEED,True,[],[]
1,SELLER_NOT_OWNER,True,[],[]
2,OVER_TRANSFER,True,[],[]
3,DOUBLE_SALE,False,[],[portion_id]
4,MUTATION_MISMATCH,True,[],[]


In [141]:
step21g_rule_df[
    "readiness_status"
] = (
    step21g_rule_df.apply(
        lambda row:
            rule_readiness_status(
                row
            ),
        axis=1
    )
)

In [142]:
step21g_rule_df[
    [
        "rule",
        "ready",
        "missing_fields",
        "missing_derived_fields",
        "readiness_status",
    ]
]

,rule,ready,missing_fields,missing_derived_fields,readiness_status
0,DUPLICATE_DEED,True,[],[],READY
1,SELLER_NOT_OWNER,True,[],[],READY
2,OVER_TRANSFER,True,[],[],READY
3,DOUBLE_SALE,False,[],[portion_id],INSUFFICIENT_VERIFIED_EVIDENCE
4,MUTATION_MISMATCH,True,[],[],READY


In [143]:
step21g_ready_rules = int(
    step21g_rule_df[
        "ready"
    ].sum()
)

step21g_blocked_rules = (
    len(
        step21g_rule_df
    )
    -
    step21g_ready_rules
)

print(
    "Ready rules:",
    step21g_ready_rules
)

print(
    "Blocked rules:",
    step21g_blocked_rules
)

Ready rules: 4
Blocked rules: 1


In [144]:
step21g_double_sale = (
    evaluate_rule_readiness(
        step21g_record,
        "DOUBLE_SALE"
    )
)

step21g_double_sale

{'rule': 'DOUBLE_SALE',
 'ready': False,
 'missing_fields': [],
 'missing_derived_fields': ['portion_id']}

In [145]:
step21g_final_gate = (
    final_transaction_gate(
        step21g_record
    )
)

step21g_final_gate

{'core_ready': True,
 'document_allowed': False,
 'final_allowed': False,
 'missing_fields': [],
 'parcel_key_ready': True}

In [146]:
step21g_execution_rows = []

for rule_name in (
    RISK_RULE_REQUIREMENTS
):

    evidence_ready = (
        evaluate_rule_readiness(
            step21g_record,
            rule_name
        )[
            "ready"
        ]
    )

    execution_allowed = (
        rule_execution_allowed(
            step21g_record,
            rule_name
        )
    )

    step21g_execution_rows.append({
        "rule":
            rule_name,

        "evidence_ready":
            evidence_ready,

        "execution_allowed":
            execution_allowed,
    })


step21g_execution_df = pd.DataFrame(
    step21g_execution_rows
)

step21g_execution_df

,rule,evidence_ready,execution_allowed
0,DUPLICATE_DEED,True,False
1,SELLER_NOT_OWNER,True,False
2,OVER_TRANSFER,True,False
3,DOUBLE_SALE,False,False
4,MUTATION_MISMATCH,True,False


In [147]:
assert (
    step21g_transaction_readiness[
        "ready"
    ]
    is True
)

assert (
    len(
        step21g_transaction_readiness[
            "missing_fields"
        ]
    )
    == 0
)

assert (
    step21g_ready_rules
    == 4
)

assert (
    step21g_blocked_rules
    == 1
)

assert (
    step21g_double_sale[
        "ready"
    ]
    is False
)

assert (
    "portion_id"
    in step21g_double_sale[
        "missing_derived_fields"
    ]
)

assert (
    step21g_final_gate[
        "core_ready"
    ]
    is True
)

assert (
    step21g_final_gate[
        "document_allowed"
    ]
    is False
)

assert (
    step21g_final_gate[
        "final_allowed"
    ]
    is False
)

assert (
    step21g_execution_df[
        "execution_allowed"
    ].sum()
    == 0
)

print(
    "Fully verified readiness tests: PASS"
)

Fully verified readiness tests: PASS


In [149]:
print("STEP 21G SUMMARY")
print("----------------")

print(
    "Transaction-core fields verified:",
    len(
        fields_to_verify
    )
)

print(
    "ParcelKey available:",
    verified_derived_field_available(
        step21g_record,
        "parcel_key"
    )
)

print(
    "Portion ID available:",
    verified_derived_field_available(
        step21g_record,
        "portion_id"
    )
)

print(
    "Transaction core ready:",
    step21g_transaction_readiness[
        "ready"
    ]
)

print(
    "Ready risk rules:",
    step21g_ready_rules
)

print(
    "Blocked risk rules:",
    step21g_blocked_rules
)

print(
    "Double-sale ready:",
    step21g_double_sale[
        "ready"
    ]
)

print(
    "Double-sale missing portion ID:",
    (
        "portion_id"
        in step21g_double_sale[
            "missing_derived_fields"
        ]
    )
)

print(
    "Document-level permission:",
    step21g_final_gate[
        "document_allowed"
    ]
)

print(
    "Final transaction allowed:",
    step21g_final_gate[
        "final_allowed"
    ]
)

print(
    "Synthetic rules actually executed:",
    int(
        step21g_execution_df[
            "execution_allowed"
        ].sum()
    )
)

print(
    "\nVerified Transaction Readiness Test: PASS"
)

STEP 21G SUMMARY
----------------
Transaction-core fields verified: 5
ParcelKey available: True
Portion ID available: False
Transaction core ready: True
Ready risk rules: 4
Blocked risk rules: 1
Double-sale ready: False
Double-sale missing portion ID: True
Document-level permission: False
Final transaction allowed: False
Synthetic rules actually executed: 0

Verified Transaction Readiness Test: PASS
